# SpanCall Lean Flow **v2.5** — Full-Data ModernBERT-large Pipeline

Essential end-to-end training and step-by-step evaluation notebook based on **v2.4.1**, stripped down to core stages and scaled to **100% training data (`DATA_FRACTION = 1.0`)**:
- **Backbone**: `answerdotai/ModernBERT-large` (native bidirectional attention, `MAX_SEQ_LEN = 4096`).
- **Model (`LeanSpanCallModel`)**: `backbone` + `ToolSelectionHead` (`H1`) + `LeanPointerHead` (`H3` over `0..C-1` candidates + `<|null|>`).
- **Decoder & Resolver (`LeanJointDecoder`, `LeanResolver`)**: Utterance-first candidate packing, cross-parameter span exclusivity (`prevent_param_collapse=True`), collapse guard (`collapse_guard=True`), and `single_pass` vs `two_step` resolution.


---
## 0. Setup & Configuration

Clones or updates the SpanCall repository, installs dependencies in editable mode, and configures full-data paths (`data/tensorized_lean_v2_5` and `results_lean_v2_5`).

In [1]:
# 0a. Pull latest code from public GitHub repository and install dependencies
import os, sys
from pathlib import Path

GIT_REPO_URL = "https://github.com/eleluong/spancall.git"
GIT_BRANCH   = "main"

cwd = Path.cwd().resolve()
if (cwd / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd
elif (cwd.parent / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd.parent
elif (cwd / "spancall" / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd / "spancall"
else:
    print(f"Cloning public repository from {GIT_REPO_URL}...")
    !git clone --branch {GIT_BRANCH} {GIT_REPO_URL}
    PROJECT_ROOT = cwd / "spancall"

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if (PROJECT_ROOT / ".git").is_dir():
    print(f"Pulling latest code from {GIT_BRANCH}...")
    !git pull origin {GIT_BRANCH} || echo "git pull note: continuing with local files"

print(f"SpanCall project root: {PROJECT_ROOT}")
%pip install -q -e ".[model,data,agent,dev]" huggingface_hub

Pulling latest code from main...
From https://github.com/eleluong/spancall
 * branch            main       -> FETCH_HEAD
Already up to date.
SpanCall project root: /content/spancall
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for spancall (pyproject.toml) ... done


In [2]:
# 0b. Configuration & Environment (Full Data)
import os, subprocess, sys
from pathlib import Path
import torch
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "spancall" / "__init__.py").is_file():
    if (PROJECT_ROOT.parent / "spancall" / "__init__.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif (PROJECT_ROOT / "spancall" / "spancall" / "__init__.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT / "spancall"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spancall.agent.backends.litellm_adapter import _load_dotenv_values

ENV = {**_load_dotenv_values(PROJECT_ROOT / ".env"), **os.environ}
HF_DATASET_ID = ENV.get("DATASET_NAME", "convoicon/spancall")
HF_TOKEN      = ENV.get("HUGGING_FACE_TOKEN") or ENV.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

DRIVE_MOUNT       = False
DRIVE_OUT         = "/content/drive/MyDrive/spancall_lean_artifacts"

DATA_DIR          = str(PROJECT_ROOT / "data" / "processed_v2")
V1_DATA_DIR       = str(PROJECT_ROOT / "data" / "processed")
TENSOR_DIR        = str(PROJECT_ROOT / "data" / "tensorized_lean_v2_5")
RESULTS_DIR       = str(PROJECT_ROOT / "results_lean_v2_5")
EVAL_SPLIT        = "hermes_eval"
TEST_SPLITS       = ["hermes_test_ood_schema", "hermes_test_ood_para", "hermes_test_adversarial"]

MAX_UTT_STALE_RATE = 0.02
MAX_UNRESOLVABLE   = 0.02
ALLOW_STALE_LABELS = False
FORCE_REALIGN      = True
FORCE_HF_DOWNLOAD  = False
MIN_JOINT_MARGIN   = 0.02

DEVICE          = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
EXPERIMENT_NAME = "spancall_lean_modernbert"
SEED            = 42

BACKBONE       = "answerdotai/ModernBERT-large"
MAX_SEQ_LEN    = 4096
MAX_SPAN_WIDTH = 16
MAX_CANDS      = 1024

_cfg_yaml = {}
try:
    with open("configs/default.yaml") as _fh:
        _cfg_yaml = yaml.safe_load(_fh) or {}
except Exception as _cfg_exc:
    print(f"[!] configs/default.yaml unreadable ({type(_cfg_exc).__name__}: {_cfg_exc})")

MAX_POS = int(((_cfg_yaml.get("optimization") or {}).get("max_pos", 3)) or 3)
_data_cfg = _cfg_yaml.get("data") or {}
MAX_TOOLS = _data_cfg.get("max_tools")
MAX_TOOLS = None if MAX_TOOLS in (None, "", 0) else int(MAX_TOOLS)
JITTER_PACK_TOOLS = bool(_data_cfg.get("jitter_pack_tools", True))

SPLIT_INPUTS = {
    "t2":  "hermes_train.jsonl",
    "dev": "hermes_dev.jsonl",
}
SAMPLE_LIMITS     = {"t2": None, "dev": None}
DATA_FRACTION     = 0.2
TENSORIZE_WORKERS = max(1, min(os.cpu_count() or 1, 16))
EVAL_LIMIT        = 500
TEST_LIMIT        = 200

print("=" * 72)
print("LEAN FLOW v2.5 CONFIG (FULL DATA)")
print("=" * 72)
print(f"EXPERIMENT_NAME   = {EXPERIMENT_NAME}  SEED = {SEED}")
print(f"BACKBONE          = {BACKBONE}")
print(f"MAX_SEQ_LEN       = {MAX_SEQ_LEN}  MAX_SPAN_WIDTH = {MAX_SPAN_WIDTH}  MAX_POS = {MAX_POS}")
print(f"DATA_FRACTION     = {DATA_FRACTION}  TENSORIZE_WORKERS = {TENSORIZE_WORKERS}")
print(f"EVAL_LIMIT        = {EVAL_LIMIT}  TEST_LIMIT = {TEST_LIMIT}")
print(f"DATA_DIR          = {DATA_DIR}")
print(f"TENSOR_DIR        = {TENSOR_DIR}")
print(f"RESULTS_DIR       = {RESULTS_DIR}")
print(f"DEVICE            = {DEVICE}")

LEAN FLOW v2.5 CONFIG (FULL DATA)
EXPERIMENT_NAME   = spancall_lean_modernbert  SEED = 42
BACKBONE          = answerdotai/ModernBERT-large
MAX_SEQ_LEN       = 4096  MAX_SPAN_WIDTH = 16  MAX_POS = 1
DATA_FRACTION     = 0.2  TENSORIZE_WORKERS = 12
EVAL_LIMIT        = 500  TEST_LIMIT = 200
DATA_DIR          = /content/spancall/data/processed_v2
TENSOR_DIR        = /content/spancall/data/tensorized_lean_v2_5
RESULTS_DIR       = /content/spancall/results_lean_v2_5
DEVICE            = cuda


In [3]:
# 0c. Runtime, GPU & Package Check (+ Optional Colab Drive Mount)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'No NVIDIA GPU detected (using MPS / CPU fallback)'

import spancall
import spancall.lean as lean
import span_template
import transformers
import datasets

print("spancall     :", spancall.__file__)
print("spancall.lean:", lean.__file__)
print("span_template:", span_template.__file__)
print("transformers :", transformers.__version__, "| datasets:", datasets.__version__)

if DRIVE_MOUNT:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print("Drive mounted:", DRIVE_OUT)

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
spancall     : /content/spancall/spancall/__init__.py
spancall.lean: /content/spancall/spancall/lean/__init__.py
span_template: /content/spancall/span_template/__init__.py
transformers : 5.22.1 | datasets: 4.8.5


---
## 1. Dataset (`data/processed_v2/`) & Pre-Flight Gate

Downloads the raw splits from Hugging Face if missing, runs `scripts/realign_v2.py` to produce render-invariant `utt_rel_start`/`utt_rel_end` offsets, and verifies the template stamp (`Gate A`) and `utterance_span` offset validity (`Gate B`).

In [4]:
from huggingface_hub import snapshot_download
from span_template.stamp import check_stamp
from spancall.lean.diagnostics import audit_gt_offsets
import json, os

CANONICAL_SPLITS = [
    "hermes_train.jsonl",
    "hermes_dev.jsonl",
    "hermes_calib.jsonl",
    "hermes_eval.jsonl",
    "hermes_test_ood_schema.jsonl",
    "hermes_test_ood_para.jsonl",
    "hermes_test_adversarial.jsonl",
]

os.makedirs(V1_DATA_DIR, exist_ok=True)
missing_v1 = [f for f in CANONICAL_SPLITS if not os.path.exists(os.path.join(V1_DATA_DIR, f))]
if FORCE_HF_DOWNLOAD or missing_v1:
    print(f"Downloading splits from Hugging Face dataset: {HF_DATASET_ID} ...")
    snapshot_download(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
        local_dir=str(PROJECT_ROOT),
        allow_patterns=["data/processed/*.jsonl", "data/processed/*.json"],
        token=HF_TOKEN,
    )

_already = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in CANONICAL_SPLITS)
if FORCE_REALIGN or not _already:
    print(f"Re-aligning v1 -> {DATA_DIR} ...")
    _realign = [
        sys.executable, os.path.join(str(PROJECT_ROOT), "scripts", "realign_v2.py"),
        "--src_dir", V1_DATA_DIR,
        "--out_dir", DATA_DIR,
        "--gate_split", EVAL_SPLIT,
        "--max_utt_stale_rate", str(MAX_UTT_STALE_RATE),
        "--max_unresolvable", str(MAX_UNRESOLVABLE),
    ]
    if ALLOW_STALE_LABELS:
        _realign.append("--force")
    _rc = subprocess.run(_realign, cwd=str(PROJECT_ROOT)).returncode
    if _rc != 0:
        raise SystemExit(f"realign_v2 exited {_rc}. Inspect data/processed_v2/realign_report.json.")
else:
    print(f"v2 splits ready in {DATA_DIR}")

# Pre-Flight Gate A: template stamp
_verdict = check_stamp(DATA_DIR)
print(f"[Gate A / Stamp] {_verdict.status}: {_verdict.message()}")
if not _verdict.ok and not ALLOW_STALE_LABELS:
    raise SystemExit("PRE-FLIGHT FAILED (template stamp): " + _verdict.message())

# Pre-Flight Gate B: offset audit on eval & dev
for _split in (EVAL_SPLIT, "hermes_dev"):
    _m = audit_gt_offsets(_split, DATA_DIR)
    _stale = _m.get("utterance_span_stale_rate")
    _unres = _m.get("unresolvable_rate")
    _ok = (
        isinstance(_stale, float) and _stale == _stale and _stale < MAX_UTT_STALE_RATE
        and ((_unres is None) or (isinstance(_unres, float) and _unres == _unres and _unres < MAX_UNRESOLVABLE))
    )
    print(f"[Gate B / Audit] {_split:<16} utterance_span_stale_rate={_stale} "
          f"({_m.get('utterance_span_targets')} targets), unresolvable_rate={_unres} -> {'PASS' if _ok else 'FAIL'}")
    if not _ok and not ALLOW_STALE_LABELS:
        raise SystemExit(f"PRE-FLIGHT FAILED on {_split}")

Re-aligning v1 -> /content/spancall/data/processed_v2 ...
[Gate A / Stamp] ok: template stamp OK (242f3c351a1d)
[Gate B / Audit] hermes_eval      utterance_span_stale_rate=0.0 (3773 targets), unresolvable_rate=0.0 -> PASS
[Gate B / Audit] hermes_dev       utterance_span_stale_rate=0.0 (187 targets), unresolvable_rate=0.0 -> PASS


---
## 2. Lean Tensorizer — Full Dataset (`DATA_FRACTION = 1.0`)

Tensorizes `hermes_train.jsonl` (`t2.jsonl`) and `hermes_dev.jsonl` (`dev.jsonl`) into 0-indexed Lean trainer rows using a shared multiprocessing fork pool.

In [5]:
from typing import Any, Dict, List, Optional
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
import json, math, multiprocessing as mp, os, queue, random, time
import yaml
from transformers import AutoTokenizer

from span_template import SPECIAL_MARKER_TOKENS
from spancall.data.schema import UnifiedSample
from spancall.model.masks import CandidateMaskBuilder
from spancall.lean.tensorize import ToolPool, tensorize_sample_lean, has_positive_targets

tokenizer = AutoTokenizer.from_pretrained(BACKBONE, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": list(SPECIAL_MARKER_TOKENS)})
mask_builder = CandidateMaskBuilder(max_seq_len=MAX_SEQ_LEN, max_span_width=MAX_SPAN_WIDTH)

PACK_TOOLS = 3
try:
    with open(os.path.join(str(PROJECT_ROOT), "configs", "default.yaml")) as _fh:
        _default_cfg = yaml.safe_load(_fh) or {}
    PACK_TOOLS = int((_default_cfg.get("data") or {}).get("pack_tools", 3) or 3)
except Exception as _exc:
    print(f"[!] configs/default.yaml unreadable ({_exc}); using PACK_TOOLS={PACK_TOOLS}")

TOOL_POOL = None
PACK_TOOLS_EPH = 1

def _iter_raw_lines(path: str):
    with open(path, "r") as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield line

def _iter_raw_dicts(path: str):
    for line in _iter_raw_lines(path):
        yield json.loads(line)

def _fraction_subset(raw_lines, frac: float, seed: int):
    raw_lines = list(raw_lines)
    if frac >= 1.0:
        return raw_lines
    rng = random.Random(int(seed))
    n = max(int(len(raw_lines) * frac), 1)
    keep = set(rng.sample(range(len(raw_lines)), min(n, len(raw_lines))))
    return [d for i, d in enumerate(raw_lines) if i in keep]

def _serialize(row):
    return {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in row.items()}

def _tensorize_lean_task(args):
    raw_line, pack_multi = args
    drops: Dict[str, int] = {}
    errors: Dict[str, int] = {}
    try:
        d = json.loads(raw_line)
        s = UnifiedSample.from_dict(d)
        ts_stats: Dict[str, Any] = {}
        row = tensorize_sample_lean(
            s,
            tokenizer=tokenizer,
            mask_builder=mask_builder,
            max_seq_len=MAX_SEQ_LEN,
            max_cands=MAX_CANDS,
            max_pos=int(MAX_POS),
            pack_multi_tools=pack_multi,
            stats=ts_stats,
            tool_pool=TOOL_POOL,
            pack_tools=PACK_TOOLS_EPH,
            max_tools=MAX_TOOLS,
            jitter=JITTER_PACK_TOOLS,
            include_abstain=True,
        )
        drops = dict(ts_stats.get("param_drop_reasons", {}))
        for _t, _cnt in (ts_stats.get("render_T_hist") or {}).items():
            _k = f"render_T={int(_t)}"
            drops[_k] = drops.get(_k, 0) + int(_cnt)
        if row is None or (not has_positive_targets(row) and not bool(row.get("is_abstain", 0))):
            return None, drops, errors
        return json.dumps(_serialize(row)), drops, errors
    except Exception as exc:
        cls = type(exc).__name__
        errors[cls] = errors.get(cls, 0) + 1
        return None, drops, errors

_WINDOW = 2048

def _drain_ordered(pool, raw, pack_multi: bool, absorb, done, rows_len, need,
                   workers: int = 1, window: int = _WINDOW):
    n = len(raw)
    guard = max(2 * int(workers), 16)
    finished_q: queue.SimpleQueue = queue.SimpleQueue()
    buffer: Dict[int, Any] = {}
    nxt = 0
    submit = 0
    inflight = 0
    while nxt < n and not done():
        budget = max(0, need - rows_len())
        cap = min(window, budget + guard)
        while submit < n and (submit - nxt) < cap and not done():
            fut = pool.submit(_tensorize_lean_task, (raw[submit], pack_multi))
            fut.add_done_callback(lambda f, i=submit: finished_q.put((i, f)))
            submit += 1
            inflight += 1
        if inflight == 0:
            break
        idx, fut = finished_q.get()
        inflight -= 1
        buffer[idx] = fut.result()
        while nxt in buffer and not done():
            absorb(buffer.pop(nxt))
            nxt += 1

def _tensorize_many(raw, pack_multi: bool, need, workers: int, executor=None):
    if need is None:
        need = len(raw)
    rows: List[str] = []
    drops_total: Counter = Counter()
    errors_total: Counter = Counter()

    def _absorb(res):
        line, drops, errors = res
        for k, v in (drops or {}).items():
            drops_total[k] += v
        for k, v in (errors or {}).items():
            errors_total[k] += v
        if line is not None and len(rows) < need:
            rows.append(line)

    def _done():
        return len(rows) >= need

    if executor is not None:
        _drain_ordered(executor, raw, pack_multi, _absorb, _done,
                       rows_len=lambda: len(rows), need=need, workers=workers)
        return rows, drops_total, errors_total

    for line in raw:
        _absorb(_tensorize_lean_task((line, pack_multi)))
        if _done():
            break
    return rows, drops_total, errors_total

os.makedirs(TENSOR_DIR, exist_ok=True)
POOL_SOURCE = SPLIT_INPUTS.get("t2") or next(iter(SPLIT_INPUTS.values()))
_pool_path = os.path.join(DATA_DIR, POOL_SOURCE)
if PACK_TOOLS > 1 and os.path.exists(_pool_path):
    TOOL_POOL = ToolPool.from_row_dicts(_iter_raw_dicts(_pool_path), seed=SEED)
    print(f"[tool-pool] built from {POOL_SOURCE}: {len(TOOL_POOL)} schemas (pack_tools={PACK_TOOLS})")
PACK_TOOLS_EPH = int(PACK_TOOLS) if TOOL_POOL is not None else 1

_fork_ctx = None
_shared_ex = None
if TENSORIZE_WORKERS > 1:
    try:
        _fork_ctx = mp.get_context("fork")
        _shared_ex = ProcessPoolExecutor(max_workers=TENSORIZE_WORKERS, mp_context=_fork_ctx)
    except Exception as _exc:
        print(f"[!] fork pool unavailable ({_exc}); running single-process")

try:
    for out_name, in_name in SPLIT_INPUTS.items():
        src = os.path.join(DATA_DIR, in_name)
        raw = _fraction_subset(_iter_raw_lines(src), DATA_FRACTION, SEED)
        limit = SAMPLE_LIMITS.get(out_name)
        t0 = time.perf_counter()
        rows, drops, errors = _tensorize_many(raw, True, limit, TENSORIZE_WORKERS, executor=_shared_ex)
        dt = time.perf_counter() - t0
        out_path = os.path.join(TENSOR_DIR, f"{out_name}.jsonl")
        with open(out_path, "w") as fh:
            fh.write("\n".join(rows) + ("\n" if rows else ""))
        render_hist = {k.split("=")[1]: v for k, v in drops.items() if k.startswith("render_T=")}
        print(f"[tensorize] {out_name:<4} <- {in_name:<18}: {len(rows):>6} rows in {dt:.1f}s "
              f"({len(rows)/max(dt, 1e-6):.1f} rows/s) | render_T={render_hist} "
              f"| match_missed={drops.get('match_missed', 0)} | errors={dict(errors)}")
finally:
    if _shared_ex is not None:
        _shared_ex.shutdown(wait=True)

fingerprint = {
    "experiment_name": EXPERIMENT_NAME,
    "seed": SEED,
    "backbone": BACKBONE,
    "max_seq_len": MAX_SEQ_LEN,
    "max_span_width": MAX_SPAN_WIDTH,
    "max_pos": MAX_POS,
    "data_fraction": DATA_FRACTION,
    "split_inputs": SPLIT_INPUTS,
    "sample_limits": SAMPLE_LIMITS,
    "lean_flow": True,
}
fp_path = os.path.join(TENSOR_DIR, "config_fingerprint.json")
with open(fp_path, "w") as fh:
    json.dump(fingerprint, fh, indent=2, sort_keys=True)
print("wrote config fingerprint ->", fp_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


[tool-pool] built from hermes_train.jsonl: 2221 schemas (pack_tools=3)
[tensorize] t2   <- hermes_train.jsonl:   4037 rows in 22.1s (182.8 rows/s) | render_T={'3': 1754, '2': 1274, '1': 1201, '4': 164, '5': 32} | match_missed=711 | errors={}
[tensorize] dev  <- hermes_dev.jsonl  :    132 rows in 0.3s (405.4 rows/s) | render_T={'2': 55, '3': 75, '1': 58, '4': 1} | match_missed=15 | errors={}
wrote config fingerprint -> /content/spancall/data/tensorized_lean_v2_5/config_fingerprint.json


---
## 3. Training — Lean Flow (`scripts/train_lean.py`)

Trains `LeanSpanCallModel` (`ModernBERT-large` + `ToolSelectionHead` H1 + `LeanPointerHead` H3) on stage `T2` with full data (`TRAIN_FRACTION = 1.0`), effective batch size 128, `bf16`, and best-checkpoint restoration.

In [6]:
import os, yaml
from scripts.train_lean import main as train_lean_main

TRAIN_FRACTION = 1.0
NUM_EPOCHS = 5
MICRO_BATCH_SIZE = 32
EFFECTIVE_BATCH_SIZE = 128

with open("configs/default.yaml") as fh:
    cfg = yaml.safe_load(fh)

cfg["experiment"]["name"] = EXPERIMENT_NAME
cfg["experiment"]["seed"] = SEED
cfg["experiment"]["output_dir"] = RESULTS_DIR
cfg["experiment"]["data_dir"] = TENSOR_DIR

cfg["optimization"]["num_epochs"] = NUM_EPOCHS
cfg["optimization"]["batch_size"] = MICRO_BATCH_SIZE
cfg["optimization"]["effective_batch_size"] = EFFECTIVE_BATCH_SIZE
cfg["optimization"]["max_tokens_per_batch"] = 4096
cfg["optimization"]["optimizer"] = "adamw"
cfg["optimization"]["scheduler"] = "cosine_warmup"
cfg["optimization"]["min_lr_ratio"] = 0.10
cfg["optimization"]["early_stopping_patience"] = 1
cfg["optimization"]["restore_best_on_exit"] = True
cfg["optimization"]["mixed_precision"] = "bf16"

cfg["training"]["gradient_accumulation_steps"] = max(int(round(EFFECTIVE_BATCH_SIZE / MICRO_BATCH_SIZE)), 1)
cfg["training"]["gradient_checkpointing"] = True
cfg["training"]["eval_splits"] = ["dev"]
cfg["training"]["eval_limit"] = 2000
cfg["training"]["freeze_encoder_stages"] = []

cfg["model"]["backbone_name"] = BACKBONE
cfg["model"]["max_seq_len"] = MAX_SEQ_LEN

_EFFECTIVE_PACK_TOOLS = int(globals().get("PACK_TOOLS_EPH", PACK_TOOLS) or 1)
cfg.setdefault("data", {})["fraction"] = TRAIN_FRACTION
cfg["data"]["pack_tools"] = _EFFECTIVE_PACK_TOOLS

CONFIG_PATH = "configs/colab_lean_v2_5.yaml"
with open(CONFIG_PATH, "w") as fh:
    yaml.safe_dump(cfg, fh)
print(f"wrote {CONFIG_PATH} (backbone={BACKBONE}, epochs={NUM_EPOCHS}, "
      f"micro_bs={MICRO_BATCH_SIZE}, effective_bs={EFFECTIVE_BATCH_SIZE}, "
      f"data_fraction={TRAIN_FRACTION}, pack_tools={_EFFECTIVE_PACK_TOOLS})")

train_lean_main([
    "--config", CONFIG_PATH,
    "--experiment_name", EXPERIMENT_NAME,
    "--output_dir", RESULTS_DIR,
    "--data_dir", TENSOR_DIR,
    "--stages", "T2",
    "--device", DEVICE,
    "--seed", str(SEED),
])

wrote configs/colab_lean_v2_5.yaml (backbone=answerdotai/ModernBERT-large, epochs=5, micro_bs=32, effective_bs=128, data_fraction=1.0, pack_tools=3)
[SpanCall Lean Train] config: configs/colab_lean_v2_5.yaml
[SpanCall Lean Train] backbone=answerdotai/ModernBERT-large


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


[lean-train:T2] start: rows=4037 epochs=5 steps/epoch=646 total_steps=3230 bs=32 accum=4 effective_bs=128 opt_steps=808 lr_floor_ratio=0.10


lean-train[T2]:   0%|          | 0/3230 [00:00<?, ?it/s]

[lean-train:T2] epoch 1/5 done: mean_loss=3.2144 (batchmean=3.1990) ptr=3.0027 tool=0.2117
[lean-eval:dev] span_acc=0.4756 (n=246) loss=1.7179 (batchmean=1.6768) ptr=1.6530 tool=0.0649 tool_acc=0.9773 (n=132) candidates/row=2.06
[lean-train:T2:epoch 1] NEW BEST on dev: exact_call=0.3106 -> saved to /content/spancall/results_lean_v2_5/spancall_lean_modernbert_seed42/best_checkpoint
[lean-train:T2] epoch 2/5 done: mean_loss=1.1685 (batchmean=1.1435) ptr=1.1619 tool=0.0066
[lean-eval:dev] span_acc=0.6098 (n=246) loss=1.3589 (batchmean=1.3163) ptr=1.3028 tool=0.0562 tool_acc=0.9773 (n=132) candidates/row=2.06
[lean-train:T2:epoch 2] NEW BEST on dev: exact_call=0.4697 -> saved to /content/spancall/results_lean_v2_5/spancall_lean_modernbert_seed42/best_checkpoint
[lean-train:T2] epoch 3/5 done: mean_loss=0.7041 (batchmean=0.6810) ptr=0.7041 tool=0.0000
[lean-eval:dev] span_acc=0.6789 (n=246) loss=1.1681 (batchmean=1.1274) ptr=1.1305 tool=0.0376 tool_acc=0.9848 (n=132) candidates/row=2.06
[le

0

In [7]:
# 3b. Training Telemetry, Best Checkpoint Selection & LeanResolver Init
import glob, json, math, os
from typing import Any, Dict, List, Optional
import torch
from transformers import AutoTokenizer

from span_template import SPECIAL_MARKER_TOKENS
from spancall.lean.model import LeanSpanCallModel
from spancall.lean.resolver import LeanResolver
from spancall.model.encoder import SpanCallTransformerEncoder
from spancall.train.config import TrainingConfig

def _num(rec, key):
    if rec is None:
        return None
    v = rec.get(key)
    if v is None:
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(f) else f

def _fmt_cell(v, width, fmt=".3f"):
    return f"{'—':>{width}}" if v is None else f"{v:>{width}{fmt}}"

exp_root = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_seed{SEED}")
log_path = os.path.join(exp_root, "train_log.jsonl")
if os.path.exists(log_path):
    with open(log_path) as fh:
        train_log = [json.loads(l) for l in fh if l.strip()]
    evals = [r for r in train_log if r.get("event") == "epoch_eval"]
    ends  = [r for r in train_log if r.get("event") == "epoch_end"]
    print("=" * 84)
    print("PER-EPOCH HELD-OUT DEV CURVE")
    print("=" * 84)
    print(f"  {'ep':>3}{'n':>6}{'span_acc':>10}{'params_ok':>11}{'exact_call':>12}"
          f"{'exact_T1':>10}{'exact_T2+':>10}{'tool_all':>10}{'tool_T2+':>10}{'train_loss':>12}{'dev_loss':>10}")
    for r in evals:
        ep = r.get("epoch", "?")
        end_r = next((e for e in ends if e.get("epoch") == ep), None)
        print(f"  {ep:>3}{int(_num(r, 'n_span') or _num(r, 'n') or 0):>6}"
              f"{_fmt_cell(_num(r, 'span_acc'), 10)}"
              f"{_fmt_cell(_num(r, 'exact_params_only'), 11)}"
              f"{_fmt_cell(_num(r, 'exact_call'), 12)}"
              f"{_fmt_cell(_num(r, 'exact_call_t1'), 10)}"
              f"{_fmt_cell(_num(r, 'exact_call_multi'), 10)}"
              f"{_fmt_cell(_num(r, 'tool_acc'), 10)}"
              f"{_fmt_cell(_num(r, 'tool_acc_multi'), 10)}"
              f"{_fmt_cell(_num(end_r, 'mean_loss'), 12, '.4f')}"
              f"{_fmt_cell(_num(r, 'eval_loss'), 10, '.4f')}")

best_cp = os.path.join(exp_root, "best_checkpoint")
if os.path.exists(os.path.join(best_cp, "model.pt")):
    CHECKPOINT_DIR = best_cp
else:
    cps = sorted(glob.glob(os.path.join(exp_root, "**", "model.pt"), recursive=True))
    CHECKPOINT_DIR = os.path.dirname(cps[-1]) if cps else None
print("\nCHECKPOINT_DIR =", CHECKPOINT_DIR)

cfg = TrainingConfig.from_yaml(CONFIG_PATH)
tokenizer = AutoTokenizer.from_pretrained(cfg.model.backbone_name, use_fast=True)
encoder = SpanCallTransformerEncoder(
    model_name_or_path=cfg.model.backbone_name,
    hidden_dropout_prob=cfg.model.dropout,
    tokenizer=tokenizer,
)
encoder.add_marker_tokens(list(SPECIAL_MARKER_TOKENS))
model = LeanSpanCallModel(
    encoder=encoder,
    head_dim=cfg.model.head_dim,
    max_span_width=cfg.model.max_span_width,
    hidden_dim=encoder.hidden_dim,
)

MAX_CALLS = int(getattr(cfg.data, "max_calls", 1))
CALL_MIN_PROB = float(getattr(cfg.data, "call_min_prob", 0.25))
CALL_MIN_PROB_RATIO = float(getattr(cfg.data, "call_min_prob_ratio", 0.5))

resolver = LeanResolver(
    model_path=CHECKPOINT_DIR,
    encoder=encoder,
    model=model,
    tokenizer=tokenizer,
    max_seq_len=cfg.model.max_seq_len,
    max_span_width=cfg.model.max_span_width,
    max_calls=MAX_CALLS,
    call_min_prob=CALL_MIN_PROB,
    call_min_prob_ratio=CALL_MIN_PROB_RATIO,
    min_joint_margin=MIN_JOINT_MARGIN,
    collapse_guard=True,
    prevent_param_collapse=True,
)
if DEVICE in ("cuda", "mps"):
    model.to(DEVICE)
model.eval()
if hasattr(encoder, "eval"):
    encoder.eval()
print("LeanResolver ready.")

PER-EPOCH HELD-OUT DEV CURVE
   ep     n  span_acc  params_ok  exact_call  exact_T1 exact_T2+  tool_all  tool_T2+  train_loss  dev_loss
    1   918     0.868      0.797       0.795     0.821     0.784     0.998     0.997      1.5314    0.4516
    2   918     0.912      0.857       0.857     0.868     0.853     1.000     1.000      0.1114    0.3723
    3   918     0.924      0.875       0.873     0.874     0.873     0.998     0.997      0.0322    0.3909
    4   918     0.919      0.873       0.873     0.868     0.875     1.000     1.000      0.0115    0.4536
    5   918     0.926      0.881       0.881     0.874     0.884     1.000     1.000      0.0047    0.4347
    1   246     0.476      0.333       0.311     0.310     0.311     0.977     0.967      1.5314    1.7179
    2   246     0.610      0.492       0.470     0.476     0.467     0.977     0.967      0.1114    1.3589
    3   246     0.679      0.553       0.538     0.524     0.544     0.985     0.978      0.0322    1.1681
    4   

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LeanResolver ready.


---
## 4. Step-by-Step Evaluation (`hermes_eval`)

### 4.0 Stratified Benchmark Slice & Step 0 (Pre-Encode Sanitize Gate)

Builds the shared deterministic stratified slice (`subset_hermes_eval.jsonl` across `F_func`, `F_json`, `A_clean`, `A_adv`) and runs the Step 0 pre-encode marker-injection check.

In [8]:
import json, os
from span_template.template import RecentRow, build_input_multi
from span_template.sanitize import SanitizeError
from spancall.data.schema import UnifiedSample
from spancall.data.align import tooldef_to_template_tool
from spancall.lean.eval_p1 import build_stratified_eval_subset, classify_eval_cohort

SUBSET_EVAL_PATH = os.path.join(RESULTS_DIR, f"subset_{EVAL_SPLIT}.jsonl")
CLEAN_EVAL_PATH  = os.path.join(RESULTS_DIR, f"subset_{EVAL_SPLIT}_clean.jsonl")
_slice_info = build_stratified_eval_subset(
    EVAL_SPLIT,
    data_dir=DATA_DIR,
    out_path=SUBSET_EVAL_PATH,
    limit=EVAL_LIMIT,
    seed=SEED,
)
print("=== 4.0 Stratified Benchmark Slice ===")
print("Available by cohort:", _slice_info["available_by_cohort"])
print("Sampled   by cohort:", _slice_info["sampled_by_cohort"], f"(total={_slice_info['sampled_total']})")

_step0_stats = {c: {"n": 0, "vetoed": 0} for c in ("F_func", "F_json", "A_clean", "A_adv")}
_clean_lines = []
with open(SUBSET_EVAL_PATH, encoding="utf-8") as _fh:
    for _line in _fh:
        _raw = _line.strip()
        if not _raw:
            continue
        _d = json.loads(_raw)
        _cohort = classify_eval_cohort(_d)
        _s = UnifiedSample.from_dict(_d)
        _st = _step0_stats.setdefault(_cohort, {"n": 0, "vetoed": 0})
        _st["n"] += 1
        _vetoed = False
        try:
            _recent = [
                RecentRow(key=str(r.get("key", "")), value=str(r.get("value", "")), source=str(r.get("source", "recent")))
                for r in (_s.recent_context or [])
            ]
            build_input_multi([tooldef_to_template_tool(t) for t in (_s.tools or [])], _s.utterance, recent_context=_recent)
        except SanitizeError:
            _vetoed = True
        except Exception:
            _vetoed = False
        if _vetoed:
            _st["vetoed"] += 1
        else:
            _clean_lines.append(_raw)

with open(CLEAN_EVAL_PATH, "w", encoding="utf-8") as _fh:
    for _l in _clean_lines:
        _fh.write(_l + "\n")

print("\n=== Step 0: Pre-Encode Sanitize Gate ===")
for _c in ("F_func", "F_json", "A_clean", "A_adv"):
    _st = _step0_stats.get(_c, {"n": 0, "vetoed": 0})
    _rate = (_st["vetoed"] / _st["n"]) if _st["n"] else 0.0
    print(f"  {_c:<8}: vetoed={_st['vetoed']:>3} / {_st['n']:<3} ({_rate:.1%})")
print(f"Non-vetoed clean eval rows -> {CLEAN_EVAL_PATH} ({len(_clean_lines)} rows)")

=== 4.0 Stratified Benchmark Slice ===
Available by cohort: {'F_func': 1485, 'F_json': 65, 'A_clean': 444, 'A_adv': 788, 'A_missing_gt_schema': 644}
Sampled   by cohort: {'F_func': 250, 'F_json': 50, 'A_clean': 150, 'A_adv': 50} (total=500)

=== Step 0: Pre-Encode Sanitize Gate ===
  F_func  : vetoed=  0 / 250 (0.0%)
  F_json  : vetoed=  0 / 50  (0.0%)
  A_clean : vetoed=  0 / 150 (0.0%)
  A_adv   : vetoed=  0 / 50  (0.0%)
Non-vetoed clean eval rows -> /content/spancall/results_lean_v2_5/subset_hermes_eval_clean.jsonl (500 rows)


---
### 4a. Step 1 in Isolation — Tool Selection (`H1`)

Scores `resolver.predict_tool` on native multi-tool rows (`len(s.tools) >= 2`), single-tool rows augmented with random pool distractors (`T=3`), and measures `F_func` vs `A_clean` score/margin separation.

In [9]:
import json, os, random, zlib
from collections import Counter, defaultdict
from spancall.data.schema import UnifiedSample
from spancall.lean.eval_p1 import classify_eval_cohort

def _pct(vals, q):
    if not vals:
        return float("nan")
    s = sorted(vals)
    idx = min(len(s) - 1, max(0, int(round((len(s) - 1) * q))))
    return s[idx]

def eval_tool_prediction_step1(subset_path: str, pool_path: str, pack_k: int = 3):
    with open(subset_path, encoding="utf-8") as fh:
        raw_dicts = [json.loads(l) for l in fh if l.strip()]

    pool = {}
    if os.path.exists(pool_path):
        with open(pool_path, encoding="utf-8") as fh:
            for l in fh:
                if not l.strip():
                    continue
                for t in (json.loads(l).get("tools") or []):
                    if t.get("name") and t["name"] not in pool:
                        pool[t["name"]] = UnifiedSample.from_dict(
                            {"sample_id": "p", "utterance": "", "tools": [t], "ground_truth_calls": []}
                        ).tools[0]
    pool_names = sorted(pool)

    native_stats = {"F_func": [0, 0], "A_clean": [0, 0], "other": [0, 0]}
    by_t = defaultdict(lambda: [0, 0])
    aug_stats = [0, 0]
    probs_by_cohort = defaultdict(list)
    margins_by_cohort = defaultdict(list)
    wrong_pairs = Counter()

    for d in raw_dicts:
        cohort = classify_eval_cohort(d)
        s = UnifiedSample.from_dict(d)
        if not s.ground_truth_calls or not s.tools:
            continue
        gt_name = s.ground_truth_calls[0].tool_name
        native_multi = len(s.tools) >= 2
        candidates = list(s.tools)
        if not native_multi and pool_names:
            seen = {t.name for t in candidates}
            salt = zlib.crc32(f"step1::{SEED}::{s.sample_id}".encode("utf-8"))
            rng = random.Random(salt)
            for pick in rng.sample(pool_names, min(len(pool_names), pack_k + 3)):
                if len(candidates) >= pack_k:
                    break
                if pick not in seen:
                    candidates.append(pool[pick])
                    seen.add(pick)
            rng.shuffle(candidates)
        if len(candidates) < 2:
            continue

        pred = resolver.predict_tool(candidates, s.utterance, s.recent_context, s.current_time_iso)
        pred_name = pred.get("tool_name")
        ok = bool(pred_name) and pred_name == gt_name

        bucket = cohort if cohort in native_stats else "other"
        if native_multi:
            native_stats[bucket][1] += 1
            native_stats[bucket][0] += int(ok)
            t_key = str(len(candidates)) if len(candidates) <= 4 else "5+"
            by_t[t_key][1] += 1
            by_t[t_key][0] += int(ok)
        else:
            aug_stats[1] += 1
            aug_stats[0] += int(ok)

        probs_by_cohort[cohort].append(float(pred.get("tool_prob") or 0.0))
        margins_by_cohort[cohort].append(float(pred.get("tool_margin") or 0.0))
        if pred_name and not ok:
            wrong_pairs[(gt_name, pred_name)] += 1

    print("=== Step 1a: Native Multi-Tool Accuracy (len(s.tools) >= 2) ===")
    tot_c = sum(v[0] for v in native_stats.values())
    tot_n = sum(v[1] for v in native_stats.values())
    print(f"  Overall native multi-tool: {tot_c / max(1, tot_n):.4f} ({tot_c}/{tot_n})")
    for bk in ("F_func", "A_clean", "other"):
        c, n = native_stats[bk]
        if n:
            print(f"    {bk:<8}: {c / n:.4f} ({c}/{n})")
    print("  By native candidate count T:", {k: f"{v[0]}/{v[1]} ({v[0]/v[1]:.3f})" for k, v in sorted(by_t.items())})

    print("\n=== Step 1b: Single-Tool + Random Pool Distractors (T=3) ===")
    if aug_stats[1]:
        print(f"  Single+random distractors: {aug_stats[0] / aug_stats[1]:.4f} ({aug_stats[0]}/{aug_stats[1]})")

    print("\n=== Step 1c: Score & Margin Separation (F_func vs A_clean) ===")
    for bk in ("F_func", "A_clean"):
        ps = probs_by_cohort.get(bk, [])
        ms = margins_by_cohort.get(bk, [])
        if ps:
            print(f"  {bk:<8} (n={len(ps):>3}) | tool_prob p10={_pct(ps,0.1):.3f} p50={_pct(ps,0.5):.3f} p90={_pct(ps,0.9):.3f} "
                  f"| margin p10={_pct(ms,0.1):.3f} p50={_pct(ms,0.5):.3f} p90={_pct(ms,0.9):.3f}")
    if wrong_pairs:
        print("  Top confusions (gt -> pred):", [(f"{g} -> {p}", cnt) for (g, p), cnt in wrong_pairs.most_common(6)])

eval_tool_prediction_step1(CLEAN_EVAL_PATH, os.path.join(DATA_DIR, f"{EVAL_SPLIT}.jsonl"))

=== Step 1a: Native Multi-Tool Accuracy (len(s.tools) >= 2) ===
  Overall native multi-tool: 0.8743 (306/350)
    F_func  : 0.9574 (225/235)
    A_clean : 0.8400 (63/75)
    other   : 0.4500 (18/40)
  By native candidate count T: {'2': '233/254 (0.917)', '3': '45/61 (0.738)', '4': '24/31 (0.774)', '5+': '4/4 (1.000)'}

=== Step 1b: Single-Tool + Random Pool Distractors (T=3) ===
  Single+random distractors: 0.9467 (142/150)

=== Step 1c: Score & Margin Separation (F_func vs A_clean) ===
  F_func   (n=250) | tool_prob p10=0.970 p50=1.000 p90=1.000 | margin p10=0.944 p50=1.000 p90=1.000
  A_clean  (n=150) | tool_prob p10=0.781 p50=0.998 p90=1.000 | margin p10=0.561 p50=0.996 p90=1.000
  Top confusions (gt -> pred): [('connect_to_alpaca_api -> get_market_data', 1), ('create_wallet -> initiate_crypto_payment', 1), ('register_new_employee -> assign_department', 1), ('start_audio_recording -> mix_audio_tracks', 1), ('track_sales_trends -> analyze_customer_buying_patterns', 1), ('extract_clai

---
### 4b. Step 2 in Isolation — Parameter Extraction (`H3`, Oracle Tool Given)

Breaks Step 2 (`resolver.try_resolve(gt_tool, ...)`) into:
- **Step 2a**: Candidate Mask Parity (`T=all` vs `T=1`).
- **Step 2b**: Oracle Candidates (`protect_parity_spans=True`, gates OFF) vs Served Candidates (`protect_parity_spans=False`).
- **Step 2c**: Gated & Schema-Validated Extraction (`collapse_guard=True`, `min_joint_margin`).

In [10]:
import math
from spancall.lean.eval_p1 import eval_param_extraction_lean
from spancall.lean.resolver import LeanResolver

def _fmt_p1(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "n/a"
    return f"{v:.4f}" if isinstance(v, float) else str(v)

_p1_2a_all = eval_param_extraction_lean(
    CLEAN_EVAL_PATH, resolver=resolver, data_dir=DATA_DIR, limit=None,
    measure_candidate_parity=True, protect_parity_spans=False,
    context_mode="all", stratum_filter="F", exclude_tools=["json_output"], quiet=True,
)
_p1_2a_t1 = eval_param_extraction_lean(
    CLEAN_EVAL_PATH, resolver=resolver, data_dir=DATA_DIR, limit=None,
    measure_candidate_parity=True, protect_parity_spans=False,
    context_mode="single", stratum_filter="F", exclude_tools=["json_output"], quiet=True,
)
print("=== Step 2a: Candidate Mask Parity (F_func, protection OFF) ===")
print(f"  T=all (multi-tool render): parity={_fmt_p1(_p1_2a_all['candidate_parity'])} "
      f"({_p1_2a_all['candidate_parity_hit']}/{_p1_2a_all['candidate_parity_checked']}) | "
      f"miss_reasons={_p1_2a_all['parity_miss_reasons']}")
print(f"  T=1   (two-step render)  : parity={_fmt_p1(_p1_2a_t1['candidate_parity'])} "
      f"({_p1_2a_t1['candidate_parity_hit']}/{_p1_2a_t1['candidate_parity_checked']}) | "
      f"miss_reasons={_p1_2a_t1['parity_miss_reasons']}")

_step2_arms = [
    ("2b. Oracle Cands (protect=T, gates=OFF)",    True,  False, 0.0,              "single"),
    ("2a+2b. Served Cands (protect=F, gates=OFF)", False, False, 0.0,              "single"),
    (f"2c. + Guards & Margin ({MIN_JOINT_MARGIN}) [T=1]",  False, True, MIN_JOINT_MARGIN, "single"),
    (f"2c-all. + Guards & Margin ({MIN_JOINT_MARGIN}) [T=all]", False, True, MIN_JOINT_MARGIN, "all"),
]

print("\n=== Step 2b & 2c: Sub-Stage Waterfall (F_func vs A_clean) ===")
print(f"{'Stage / Config':<44}{'Parity':>8}{'Prec':>8}{'Rec':>8}{'F1':>8}{'Coll_F':>8}{'Exact_F':>9}{'A_Hit':>8}")
print("-" * 101)
for _label, _prot, _cg, _mjm, _cmode in _step2_arms:
    _r_arm = LeanResolver(
        model=model, encoder=encoder, tokenizer=tokenizer,
        max_seq_len=cfg.model.max_seq_len, max_span_width=cfg.model.max_span_width,
        max_calls=MAX_CALLS, call_min_prob=CALL_MIN_PROB, call_min_prob_ratio=CALL_MIN_PROB_RATIO,
        min_joint_margin=_mjm, collapse_guard=_cg, prevent_param_collapse=_cg,
    )
    _mf = eval_param_extraction_lean(
        CLEAN_EVAL_PATH, resolver=_r_arm, data_dir=DATA_DIR, limit=None,
        measure_candidate_parity=True, protect_parity_spans=_prot,
        context_mode=_cmode, stratum_filter="F", exclude_tools=["json_output"], quiet=True,
    )
    _ma = eval_param_extraction_lean(
        CLEAN_EVAL_PATH, resolver=_r_arm, data_dir=DATA_DIR, limit=None,
        measure_candidate_parity=False, protect_parity_spans=False,
        context_mode=_cmode, stratum_filter="A", exclude_tools=["json_output"], quiet=True,
    )
    _a_hit = (_ma["extracted"] / _ma["n"]) if _ma["n"] else float("nan")
    print(f"{_label:<44}{_fmt_p1(_mf['candidate_parity']):>8}"
          f"{_fmt_p1(_mf['extraction_precision']):>8}{_fmt_p1(_mf['extraction_recall']):>8}"
          f"{_fmt_p1(_mf['extraction_f1']):>8}{_fmt_p1(_mf['pointer_collapse_rate_F']):>8}"
          f"{_fmt_p1(_mf['exact_payload_match_F']):>9}{_fmt_p1(_a_hit):>8}")

=== Step 2a: Candidate Mask Parity (F_func, protection OFF) ===
  T=all (multi-tool render): parity=0.3418 (566/1656) | miss_reasons={'truncated': 821, 'never_generated_typefilter': 235, 'never_generated_width': 34}
  T=1   (two-step render)  : parity=0.7615 (1261/1656) | miss_reasons={'truncated': 303, 'never_generated_typefilter': 64, 'never_generated_width': 28}

=== Step 2b & 2c: Sub-Stage Waterfall (F_func vs A_clean) ===
Stage / Config                                Parity    Prec     Rec      F1  Coll_F  Exact_F   A_Hit
-----------------------------------------------------------------------------------------------------
2b. Oracle Cands (protect=T, gates=OFF)       0.9251  0.5984  0.9956  0.7475  0.1600   0.4480  0.7836
2a+2b. Served Cands (protect=F, gates=OFF)    0.7615  0.5367  0.9951  0.6974  0.2040   0.4200  0.7836
2c. + Guards & Margin (0.02) [T=1]            0.7615  0.5826  0.9844  0.7320  0.0120   0.4360  0.7836
2c-all. + Guards & Margin (0.02) [T=all]      0.3418  0.583

---
### 4c. Step 3 — Composed E2E (`single_pass` vs `two_step`), Stage Waterfall & Headline Table

Runs E2E inference on `SUBSET_EVAL_PATH` in both `single_pass` and `two_step` modes, prints the stage-by-stage waterfall funnel, and compares `B1a (regex)` vs `SpanCall Lean (single_pass)` vs `SpanCall Lean (two_step)` via `scripts/eval_toolcall.py`.

In [11]:
import json, os, subprocess, sys
from typing import Any, Dict, List, Optional
from spancall.data.schema import UnifiedSample
from spancall.eval.toolcall.evaluator import payload_equal
from spancall.lean.eval_p1 import _parity_spans_for_call, classify_eval_cohort, predicted_call_count as _n_predicted_calls

def _param_detail(pr) -> Dict[str, Any]:
    return {
        "value": getattr(pr, "value", None),
        "confidence": float(getattr(pr, "confidence", 0.0) or 0.0),
        "margin": float(getattr(pr, "margin", 0.0) or 0.0),
        "source_type": getattr(pr, "source_type", None),
        "char_span": list(pr.char_span) if getattr(pr, "char_span", None) else None,
        "p_abstain": 0.0,
    }

def _features_detail(features) -> Dict[str, Any]:
    feats = dict(features or {})
    if "tool_scores" in feats and isinstance(feats["tool_scores"], dict):
        top = sorted(feats["tool_scores"].items(), key=lambda kv: -float(kv[1]))[:3]
        feats["tool_scores"] = {k: float(v) for k, v in top}
    return {k: v for k, v in feats.items() if k not in ("recent_selected", "tool_scores_all")}

def predict_split(split_name: str, out_path: str, limit: Optional[int] = None,
                  detailed: bool = True, protect_parity_spans: bool = False,
                  resolve_mode: Optional[str] = None, src_path: Optional[str] = None) -> None:
    sub_src = os.path.join(RESULTS_DIR, f"subset_{split_name}.jsonl")
    src = src_path or (sub_src if (split_name == EVAL_SPLIT and os.path.exists(sub_src)) else os.path.join(DATA_DIR, f"{split_name}.jsonl"))
    if not os.path.exists(src):
        print(f"no source {src} — skipped")
        return
    preds: List[Dict[str, Any]] = []
    subset_samples: List[Dict[str, Any]] = []
    n = 0
    parity_checked = 0
    parity_hit = 0
    collapse_hits = 0
    _supports_parity = bool(getattr(resolver, "supports_candidate_parity", False))
    with open(src) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            raw_dict = json.loads(line)
            s = UnifiedSample.from_dict(raw_dict)
            _p_spans = None
            if _supports_parity and s.ground_truth_calls:
                _gt0 = s.ground_truth_calls[0]
                _raw_ps = _parity_spans_for_call(s, 0)
                if _raw_ps:
                    _p_spans = {f"{_gt0.tool_name}::{k}": v for k, v in _raw_ps.items()}
            if _supports_parity:
                res = resolver.try_resolve_candidates(
                    s.tools, s.utterance, s.recent_context, s.current_time_iso,
                    parity_spans=_p_spans, protect_parity_spans=protect_parity_spans,
                    resolve_mode=resolve_mode,
                )
            else:
                res = resolver.try_resolve_candidates(s.tools, s.utterance, s.recent_context, s.current_time_iso)
            _feats = getattr(res, "features", None) or {}
            parity_checked += int(_feats.get("parity_checked", 0) or 0)
            parity_hit += int(_feats.get("parity_hit", 0) or 0)
            if res.hit and _feats.get("pointer_collapsed"):
                collapse_hits += 1
            rec = {
                "sample_id": s.sample_id,
                "stratum": s.stratum.value,
                "gt_tool": (s.ground_truth_calls[0].tool_name if s.ground_truth_calls else None),
                "gt_args": (s.ground_truth_calls[0].arguments if s.ground_truth_calls else {}),
                "gt_calls": [{"tool_name": c.tool_name, "arguments": dict(c.arguments or {})}
                             for c in (s.ground_truth_calls or [])],
                "n_gt_calls": len(s.ground_truth_calls or []),
                "calls": [c.as_dict() for c in (getattr(res, "calls", None) or [])],
                "n_pred_calls": _n_predicted_calls(res),
                "hit": bool(res.hit),
                "status": getattr(res, "status", "hit" if res.hit else "partial"),
                "payload": res.payload,
                "payload_partial": getattr(res, "payload_partial", {}),
                "missing_required": getattr(res, "missing_required", []),
                "missing_optional": getattr(res, "missing_optional", []),
                "tool_name": res.tool_name,
                "confidence": float(getattr(res, "confidence", res.accept_score) or 0.0),
                "accept_probability": float(res.accept_score or 0.0),
                "joint_margin": float(getattr(res, "joint_margin", 0.0) or 0.0),
                "abstain_reason": res.abstain_reason,
                "latency_ms": float(getattr(res, "latency_ms", 0.0) or 0.0),
            }
            if detailed:
                rec["parameters"] = {p: _param_detail(pr) for p, pr in (res.parameters or {}).items()}
                rec["features"] = _features_detail(getattr(res, "features", {}))
            preds.append(rec)
            subset_samples.append(raw_dict)
            n += 1
            if limit is not None and n >= limit:
                break
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as fh:
        for p in preds:
            fh.write(json.dumps(p, default=str) + "\n")
    if os.path.abspath(src) != os.path.abspath(sub_src):
        with open(sub_src, "w") as fh:
            for s_dict in subset_samples:
                fh.write(json.dumps(s_dict, default=str) + "\n")
    n_hit = sum(1 for p in preds if p["hit"])
    _par_str = f"{parity_hit}/{parity_checked} ({parity_hit / parity_checked:.3f})" if parity_checked else "n/a"
    print(f"predicted {n} rows ({resolve_mode or 'default'}) -> {out_path} (hit={n_hit}, collapse_hits={collapse_hits}, e2e_parity={_par_str})")

_p_single = os.path.join(RESULTS_DIR, "preds_eval.jsonl")
_p_two    = os.path.join(RESULTS_DIR, "preds_eval_twostep.jsonl")
predict_split(EVAL_SPLIT, _p_single, limit=None, resolve_mode="single_pass", src_path=SUBSET_EVAL_PATH)
predict_split(EVAL_SPLIT, _p_two,    limit=None, resolve_mode="two_step",    src_path=SUBSET_EVAL_PATH)

_raw_sub = {}
with open(SUBSET_EVAL_PATH) as _fh:
    for _l in _fh:
        if _l.strip():
            _d = json.loads(_l)
            _raw_sub[_d["sample_id"]] = (_d, UnifiedSample.from_dict(_d))

def _waterfall(preds_path: str, label: str) -> None:
    with open(preds_path) as _fh:
        _ps = [json.loads(l) for l in _fh if l.strip()]
    _f  = [p for p in _ps if classify_eval_cohort(_raw_sub[p["sample_id"]][0]) == "F_func"]
    _ac = [p for p in _ps if classify_eval_cohort(_raw_sub[p["sample_id"]][0]) == "A_clean"]
    _aa = [p for p in _ps if classify_eval_cohort(_raw_sub[p["sample_id"]][0]) == "A_adv"]

    n_tot = len(_f)
    s0 = [p for p in _f if not str(p.get("abstain_reason") or "").startswith("sanitize")]
    s1 = [p for p in s0 if p.get("tool_name") == p.get("gt_tool")]
    s2a = [p for p in s1 if int((p.get("features") or {}).get("parity_checked") or 0) == int((p.get("features") or {}).get("parity_hit") or 0)]
    s2b_exact = []
    for p in s2a:
        _s = _raw_sub[p["sample_id"]][1]
        _gt_t = next((t for t in _s.tools if t.name == p["gt_tool"]), None)
        if payload_equal(p.get("gt_args") or {}, p.get("payload_partial") or {}, _gt_t):
            s2b_exact.append(p)
    s2c_hit = [p for p in s2b_exact if p.get("hit")]

    print(f"\n--- Waterfall Funnel [{label}] on F_func (N={n_tot}) ---")
    for _name, _cnt in [
        ("Total F_func", n_tot),
        ("-> Step 0: Sanitize OK", len(s0)),
        ("-> Step 1: Correct Tool", len(s1)),
        ("-> Step 2a: All GT Spans in C=512", len(s2a)),
        ("-> Step 2b: All Params Matched (partial)", len(s2b_exact)),
        ("-> Step 2c: Schema & Gates Hit (E2E Exact)", len(s2c_hit)),
    ]:
        print(f"  {_name:<44} {_cnt:>4} / {n_tot} ({_cnt / max(1, n_tot):>6.1%})")
    _ac_hit = sum(1 for p in _ac if p.get("hit"))
    _aa_san = sum(1 for p in _aa if str(p.get("abstain_reason") or "").startswith("sanitize"))
    print(f"  [A_clean] correct abstain: {len(_ac) - _ac_hit}/{len(_ac)} ({1.0 - _ac_hit / max(1, len(_ac)):.1%}) | "
          f"[A_adv] sanitize-vetoed: {_aa_san}/{len(_aa)} ({_aa_san / max(1, len(_aa)):.1%})")

_waterfall(_p_single, "single_pass")
_waterfall(_p_two, "two_step")

# Evaluate B1a, single_pass, and two_step via eval_toolcall.py
for _args, _out in [
    (["--baseline", "b1a"], f"toolcall_metrics_{EVAL_SPLIT}_b1a.json"),
    (["--predictions", _p_single], f"toolcall_metrics_{EVAL_SPLIT}_lean.json"),
    (["--predictions", _p_two], f"toolcall_metrics_{EVAL_SPLIT}_lean_twostep.json"),
]:
    subprocess.run([
        sys.executable, "scripts/eval_toolcall.py",
        "--test_data", SUBSET_EVAL_PATH,
        *_args,
        "--output", os.path.join(RESULTS_DIR, _out),
    ], check=True)

def _load_metrics(path):
    if not os.path.exists(path):
        return None
    with open(path) as fh:
        return json.load(fh)

print("\n=== Headline Comparison Table (hermes_eval) ===")
_headers = ["system", "n_F", "exact_F", "set_exact_F", "count_acc", "hit_F", "collapse_F", "param_F1_F", "false_hit_on_A", "exact_all"]
print(" | ".join(_headers))
print("-|" * len(_headers))
for _label, _fn in [
    ("B1a (regex)", f"toolcall_metrics_{EVAL_SPLIT}_b1a.json"),
    ("SpanCall Lean (single_pass)", f"toolcall_metrics_{EVAL_SPLIT}_lean.json"),
    ("SpanCall Lean (two_step)", f"toolcall_metrics_{EVAL_SPLIT}_lean_twostep.json"),
]:
    _m = _load_metrics(os.path.join(RESULTS_DIR, _fn))
    if not _m:
        continue
    _mf = _m.get("metrics_by_stratum", {}).get("F", {})
    _row = {
        "system": _label,
        "n_F": int(_mf.get("n", 0)),
        "exact_F": round(_mf.get("exact_payload_match", 0.0), 3),
        "set_exact_F": round(_mf.get("call_exact_set_match", 0.0), 3),
        "count_acc": round(_m.get("call_count_acc", 0.0), 3),
        "hit_F": round(_mf.get("hit_rate", 0.0), 3),
        "collapse_F": round(_mf.get("collapse_hit_rate", 0.0), 3),
        "param_F1_F": round(_mf.get("param_f1", 0.0), 3),
        "false_hit_on_A": round(_m.get("false_hit_rate_on_a", 0.0), 4),
        "exact_all": round(_m.get("exact_payload_match", 0.0), 3),
    }
    print(" | ".join(str(_row[h]) for h in _headers))

predicted 500 rows (single_pass) -> /content/spancall/results_lean_v2_5/preds_eval.jsonl (hit=354, collapse_hits=0, e2e_parity=754/1819 (0.415))
predicted 500 rows (two_step) -> /content/spancall/results_lean_v2_5/preds_eval_twostep.jsonl (hit=361, collapse_hits=0, e2e_parity=1451/1819 (0.798))

--- Waterfall Funnel [single_pass] on F_func (N=250) ---
  Total F_func                                  250 / 250 (100.0%)
  -> Step 0: Sanitize OK                        250 / 250 (100.0%)
  -> Step 1: Correct Tool                       240 / 250 ( 96.0%)
  -> Step 2a: All GT Spans in C=512              42 / 250 ( 16.8%)
  -> Step 2b: All Params Matched (partial)       27 / 250 ( 10.8%)
  -> Step 2c: Schema & Gates Hit (E2E Exact)     21 / 250 (  8.4%)
  [A_clean] correct abstain: 78/150 (52.0%) | [A_adv] sanitize-vetoed: 25/50 (50.0%)

--- Waterfall Funnel [two_step] on F_func (N=250) ---
  Total F_func                                  250 / 250 (100.0%)
  -> Step 0: Sanitize OK             

---
### 4d. Stage-Attributed Error Forensics

Groups `F_func` misses by the exact stage where they failed (`Step 1 WRONG_TOOL`, `Step 2a PARITY_MISS`, `Step 2b POINTER_RANK_MISS`, `Step 2c GATE_OR_SCHEMA_REJECT`).

In [12]:
import json, os, textwrap
from collections import defaultdict
from spancall.data.schema import UnifiedSample
from spancall.eval.toolcall.evaluator import payload_equal
from spancall.lean.eval_p1 import classify_eval_cohort

PREDS_PATH = os.path.join(RESULTS_DIR, "preds_eval_twostep.jsonl")
if not os.path.exists(PREDS_PATH):
    PREDS_PATH = os.path.join(RESULTS_DIR, "preds_eval.jsonl")

if os.path.exists(PREDS_PATH) and os.path.exists(SUBSET_EVAL_PATH):
    with open(PREDS_PATH) as fh:
        preds = [json.loads(l) for l in fh if l.strip()]
    subs, raw_map = {}, {}
    with open(SUBSET_EVAL_PATH) as fh:
        for l in fh:
            if l.strip():
                d = json.loads(l)
                raw_map[d["sample_id"]] = d
                subs[d["sample_id"]] = UnifiedSample.from_dict(d)

    buckets = defaultdict(list)
    for p in preds:
        sid = p.get("sample_id")
        if sid not in raw_map or classify_eval_cohort(raw_map[sid]) != "F_func":
            continue
        feats = p.get("features") or {}
        if str(p.get("abstain_reason") or "").startswith("sanitize"):
            buckets["STEP0_SANITIZE"].append(p)
        elif p.get("tool_name") != p.get("gt_tool"):
            buckets["STEP1_WRONG_TOOL"].append(p)
        elif int(feats.get("parity_hit") or 0) < int(feats.get("parity_checked") or 0):
            buckets["STEP2A_PARITY_MISS"].append(p)
        else:
            s = subs[sid]
            gt_t = next((t for t in s.tools if t.name == p["gt_tool"]), None)
            partial_ok = payload_equal(p.get("gt_args") or {}, p.get("payload_partial") or {}, gt_t)
            if p.get("hit") and payload_equal(p.get("gt_args") or {}, p.get("payload") or {}, gt_t):
                buckets["EXACT_HIT"].append(p)
            elif not partial_ok:
                buckets["STEP2B_POINTER_RANK_MISS"].append(p)
            else:
                buckets["STEP2C_GATE_OR_SCHEMA_REJECT"].append(p)

    tot_f = sum(len(v) for v in buckets.values())
    print(f"F_func Stage Attribution ({os.path.basename(PREDS_PATH)}, N={tot_f}):")
    for k in ("EXACT_HIT", "STEP0_SANITIZE", "STEP1_WRONG_TOOL", "STEP2A_PARITY_MISS", "STEP2B_POINTER_RANK_MISS", "STEP2C_GATE_OR_SCHEMA_REJECT"):
        c = len(buckets[k])
        print(f"  {k:<30}{c:>4}  ({c / max(1, tot_f):>6.1%})")

F_func Stage Attribution (preds_eval_twostep.jsonl, N=250):
  EXACT_HIT                      109  ( 43.6%)
  STEP0_SANITIZE                   0  (  0.0%)
  STEP1_WRONG_TOOL                10  (  4.0%)
  STEP2A_PARITY_MISS              92  ( 36.8%)
  STEP2B_POINTER_RANK_MISS        34  ( 13.6%)
  STEP2C_GATE_OR_SCHEMA_REJECT     5  (  2.0%)


---
## 5. OOD / Adversarial Test Splits & Drive Persistence

Evaluates `LeanResolver` and `B1a` on `hermes_test_ood_schema`, `hermes_test_ood_para`, and `hermes_test_adversarial`, and optionally mirrors artifacts to Google Drive.

In [13]:
import json, os, shutil, subprocess, sys

for split in TEST_SPLITS:
    predict_split(split, os.path.join(RESULTS_DIR, f"preds_{split}.jsonl"), limit=TEST_LIMIT, resolve_mode="two_step")
    data_file = os.path.join(RESULTS_DIR, f"subset_{split}.jsonl")
    if not os.path.exists(data_file):
        data_file = os.path.join(DATA_DIR, f"{split}.jsonl")
    for label, extra in [("b1a", ["--baseline", "b1a"]), ("lean", ["--predictions", os.path.join(RESULTS_DIR, f"preds_{split}.jsonl")])]:
        subprocess.run([
            sys.executable, "scripts/eval_toolcall.py",
            "--test_data", data_file,
            *extra,
            "--output", os.path.join(RESULTS_DIR, f"toolcall_metrics_{split}_{label}.json"),
        ], check=False)

print(f"\n{'split':<26}{'system':<12}{'n_F':>6}{'exact_F':>9}{'hit_F':>8}{'param_F1':>9}{'exact_all':>10}")
print("-" * 80)
for split in TEST_SPLITS + [EVAL_SPLIT]:
    for label in ("b1a", "lean"):
        m = _load_metrics(os.path.join(RESULTS_DIR, f"toolcall_metrics_{split}_{label}.json"))
        if not m:
            continue
        m_f = m.get("metrics_by_stratum", {}).get("F", {})
        print(f"{split:<26}{label:<12}{int(m_f.get('n', 0)):>6}"
              f"{m_f.get('exact_payload_match', 0.0):>9.3f}{m_f.get('hit_rate', 0.0):>8.3f}"
              f"{m_f.get('param_f1', 0.0):>9.3f}{m.get('exact_payload_match', 0.0):>10.3f}")

if DRIVE_MOUNT:
    os.makedirs(DRIVE_OUT, exist_ok=True)
    for name, src in [("data_tensorized_lean_v2_5", TENSOR_DIR), ("results_lean_v2_5", RESULTS_DIR)]:
        if os.path.isdir(src):
            dst = os.path.join(DRIVE_OUT, name)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"mirrored {src} -> {dst}")

predicted 200 rows (two_step) -> /content/spancall/results_lean_v2_5/preds_hermes_test_ood_schema.jsonl (hit=118, collapse_hits=0, e2e_parity=358/461 (0.777))
predicted 200 rows (two_step) -> /content/spancall/results_lean_v2_5/preds_hermes_test_ood_para.jsonl (hit=177, collapse_hits=0, e2e_parity=363/545 (0.666))
predicted 200 rows (two_step) -> /content/spancall/results_lean_v2_5/preds_hermes_test_adversarial.jsonl (hit=77, collapse_hits=0, e2e_parity=n/a)

split                     system         n_F  exact_F   hit_F param_F1 exact_all
--------------------------------------------------------------------------------
hermes_test_ood_schema    b1a             31    0.065   0.000    0.000     0.010
hermes_test_ood_schema    lean            31    0.581   0.806    0.667     0.130
hermes_test_ood_para      b1a            200    0.010   0.020    0.018     0.010
hermes_test_ood_para      lean           200    0.655   0.885    0.835     0.655
hermes_test_adversarial   b1a              0    0.